# Módulo 1 · Ingesta Programada cada 30 minutos

**Objetivo:** Extraer en una sola petición incremental las transacciones nuevas desde la API partiendo de `MAX(id) + 1` y almacenarlas **sin transformar** en Unity Catalog Volumes, deteniéndose inmediatamente tral momento de guardar el lote.


In [0]:
%run ./000_Setup

In [0]:
# AMBIENTE PRUEBAS LIMPIAR LOS DATOS

# 1. Eliminar carpetas de datos crudos, checkpoints y esquemas en Unity Catalog Volume
'''
# AMBIENTE PRUEBAS LIMPIAR LOS DATOS

# 1. Eliminar carpetas de datos crudos, checkpoints y esquemas en Unity Catalog Volume
for carpeta in ["uses", "checkpoints", "schemas"]:
    ruta = f"{BASE}/{carpeta}"
    dbutils.fs.rm(ruta, recurse=True)
    print(f"✔ Carpeta eliminada: {ruta}")

# 2. Eliminar tablas Delta de transacciones para que nazcan limpias desde la versión 0
spark.sql(f"DROP TABLE IF EXISTS {qualified_name(BRONZE_USES)}")
spark.sql(f"DROP TABLE IF EXISTS {qualified_name(SILVER_USES)}")
spark.sql(f"DROP TABLE IF EXISTS {qualified_name(GOLD_USES_TX)}")
print("✔ Tablas Bronze, Silver y Gold (uses) eliminadas correctamente.")
'''

## Paso 1 · Configuración de Parámetros de Ejecución (Ventana 30 min)

In [0]:
import requests
import json

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from pathlib import Path

ahora_bogota      = datetime.now(ZONA_HORARIA)
fecha_hoy         = ahora_bogota.date()
trx_id_inicial    = get_max_id(BRONZE_USES) + 1
timestamp_corrida = ahora_bogota.strftime("%Y%m%d_%H%M")
carpeta_dia       = Path(RAW_PATH) / f"{fecha_hoy:%Y}" / f"{fecha_hoy:%m}" / f"{fecha_hoy:%d}"
nombre_archivo    = f"uses_{timestamp_corrida}.json"
destino           = carpeta_dia / nombre_archivo

def crear_sesion_con_reintentos(reintentos: int = 3, backoff: float = 4.0) -> requests.Session:
    estrategia = Retry(
        total=reintentos,
        read=reintentos,
        connect=reintentos,
        backoff_factor=backoff,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"]
    )
    adaptador = HTTPAdapter(max_retries=estrategia)
    sesion = requests.Session()
    sesion.mount("https://", adaptador)
    sesion.mount("http://", adaptador)
    return sesion

print(f"Hora ejecución (Bogotá) : {ahora_bogota:%Y-%m-%d %H:%M:%S}")
print(f"Archivo destino         : {destino}")
print(f"Transaction ID inicial  : {trx_id_inicial}")

## Paso 2 · Extracción Única Incremental a RAW


In [0]:
api_key = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
headers = {"apikey": api_key}

total_registros = 0
ultimo_id_lote  = None
proximo_trx_id  = trx_id_inicial

if destino.exists():
    print(f"⏭ Omitido: El archivo {nombre_archivo} ya existe en RAW para este minuto.")
else:
    params = {
        "ProjectID": API_PROJECT_ID,
        "TransactionID": trx_id_inicial
    }

    print(f"Consultando TransactionID={trx_id_inicial}...")

    with crear_sesion_con_reintentos() as session:
        try:
            resp = session.get(API_ENDPOINT_DEFAULT, headers=headers, params=params, timeout=API_TIMEOUT_DEFAULT)
            resp.raise_for_status()
        except requests.exceptions.HTTPError as e:
            print(f"Error HTTP consultando ID={trx_id_inicial}: {e}")
            if e.response is not None:
                print(f"   Detalle servidor ({e.response.status_code}): {e.response.text[:300]}")
            raise

    payload       = resp.json()
    data_obj      = payload.get("data") or {}
    transacciones = data_obj.get("transactions") or []
    cantidad      = len(transacciones)

    if cantidad == 0:
        print(f"✔ Al día: No hay transacciones nuevas desde el ID {trx_id_inicial}. Proceso finalizado.")
    else:
        carpeta_dia.mkdir(parents=True, exist_ok=True)
        destino.write_bytes(resp.content)

        total_registros = cantidad
        ultimo_id_lote  = max(int(tx["id"]) for tx in transacciones)
        proximo_trx_id  = ultimo_id_lote + 1

        print(f"✔ Guardado {nombre_archivo} | Registros: {cantidad:,} | Rango ID: {trx_id_inicial} -> {ultimo_id_lote} | Peso: {round(destino.stat().st_size / 1024, 2)} KB")
        print("✔ Lote descargado correctamente. Deteniendo ingesta hasta la próxima ventana programada.")

resumen_corrida = {
    "timestamp_corrida": timestamp_corrida,
    "archivo_generado": str(destino) if total_registros > 0 else None,
    "trx_id_inicial": trx_id_inicial,
    "ultimo_id_lote": ultimo_id_lote,
    "proximo_trx_id": proximo_trx_id,
    "registros_descargados": total_registros
}

print("\n" + "=" * 60)
print(f"RESUMEN INGESTA RAW ({timestamp_corrida})")
print(f"  Registros descargados : {resumen_corrida['registros_descargados']:,}")
print(f"  Rango de IDs          : {trx_id_inicial} -> {ultimo_id_lote}")
print(f"  Próximo ID (siguiente): {resumen_corrida['proximo_trx_id']}")
print("=" * 60)

## Paso 3 · Verificar los Archivos Descargados en RAW

In [0]:
archivos = spark.read.format("binaryFile").load(RAW_GLOB)
display(archivos.select("path", "length", "modificationTime").orderBy(F.col("modificationTime").desc()).limit(20))

In [0]:
print("Total archivos en RAW :", archivos.count())
print("Tamaño total carpeta  :", size_mb(RAW_PATH), "MB")

In [0]:
# Publicar valor para la tarea If/else del Databricks Workflow
dbutils.jobs.taskValues.set(
    key="registros_descargados",
    value=int(resumen_corrida["registros_descargados"])
)

dbutils.notebook.exit(json.dumps(resumen_corrida))